**Part 1 — Data Understanding & Preprocessing**

In [33]:
!pip install langchain langchain-google-genai langchain-community langchain-core pypdf faiss-cpu

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
import getpass

In [19]:
os.environ["GEMINI_API_KEY"]=getpass.getpass("Enter your API Key")

Enter your API Key··········


In [20]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.6-flash'
)

In [21]:
from langchain_community.document_loaders import PyPDFLoader #helps in reading the source data which is a pdf file

**1. Load the PDF document**

In [22]:
loader=PyPDFLoader("/content/intro-to-ml.pdf")

**2. Extract text from all pages **

In [23]:
documents=loader.load()


3.Explore document structure:
a. Number of pages
b. Text quality issues etc.

In [24]:
report=""

for page in documents:
  report += page.page_content + '\n'

print("Total number of pages",len(documents))

Total number of pages 392


In [25]:
import re

def check_text_quality(documents, min_chars=50, max_non_word_ratio=0.35):
    issues_by_page = {}
    line_counts = {}

    # Count recurring lines across pages to spot headers/footers
    for doc in documents:
        unique_lines = {line.strip() for line in doc.page_content.splitlines() if len(line.strip()) > 3}
        for line in unique_lines:
            line_counts[line] = line_counts.get(line, 0) + 1

    header_footer_threshold = max(3, int(len(documents) * 0.6))

    for page_num, doc in enumerate(documents, start=1):
        text = doc.page_content or ""
        stripped = text.strip()
        page_issues = []

        # 1. Empty or near-empty page (often scanned images or blanks)
        if len(stripped) < min_chars:
            page_issues.append(f"Very low text ({len(stripped)} chars) - possible scan or blank page")

        # 2. Replacement / control characters (encoding or PDF font-mapping failure)
        if "\ufffd" in text or re.search(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", text):
            page_issues.append("Contains replacement () or control characters")

        # 3. High ratio of non-alphanumeric/non-space characters (garbled OCR / symbol noise)
        if stripped:
            noisy_chars = len(re.findall(r"[^\w\s.,;:!?\"'()-]", stripped))
            ratio = noisy_chars / len(stripped)
            if ratio > max_non_word_ratio:
                page_issues.append(f"High symbol/noise ratio ({ratio:.0%})")

        # 4. Broken word spacing or excessive single-character tokens (OCR / PDF layout artifact)
        tokens = stripped.split()
        if len(tokens) > 15:
            single_char_ratio = sum(1 for t in tokens if len(t) == 1 and t.isalnum()) / len(tokens)
            if single_char_ratio > 0.25:
                page_issues.append(f"Many split/single-char words ({single_char_ratio:.0%})")

        # 5. Repeated lines across most pages (headers/footers leaking into body text)
        repeated = [
            line for line in {l.strip() for l in text.splitlines() if len(l.strip()) > 3}
            if line_counts.get(line, 0) >= header_footer_threshold
        ]
        if repeated:
            page_issues.append(f"Repeated header/footer lines ({len(repeated)}): {repeated[:2]}")

        if page_issues:
            issues_by_page[page_num] = page_issues

    return issues_by_page


quality_report = check_text_quality(documents)

print(f"Total pages: {len(documents)} | Pages with potential issues: {len(quality_report)}")
for page_num, problems in quality_report.items():
    print(f"\nPage {page_num}:")
    for p in problems:
        print(f"  - {p}")

Total pages: 392 | Pages with potential issues: 13

Page 2:
  - Very low text (0 chars) - possible scan or blank page

Page 31:
  - Many split/single-char words (33%)

Page 139:
  - Many split/single-char words (31%)

Page 140:
  - Many split/single-char words (26%)

Page 144:
  - Very low text (0 chars) - possible scan or blank page

Page 185:
  - Many split/single-char words (44%)

Page 194:
  - Many split/single-char words (76%)

Page 224:
  - Very low text (0 chars) - possible scan or blank page

Page 247:
  - Many split/single-char words (52%)

Page 269:
  - Many split/single-char words (43%)

Page 311:
  - Many split/single-char words (25%)

Page 318:
  - Very low text (0 chars) - possible scan or blank page

Page 336:
  - Very low text (0 chars) - possible scan or blank page


**4. Split text into chunks**  

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter #helps in chunking

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000, #inside each chunk, how many tokens we have ?
    chunk_overlap=150 #how many tokens are overlapping between each chunk ?
)

chunks=text_splitter.split_documents(documents)

**Part 2 — Embedding & Vector Database **
Build the retrieval foundation of the RAG system.

**1. Generate embeddings for all text chunks  **

In [27]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings1=GoogleGenerativeAIEmbeddings(
    model='gemini-embedding-2'
)

### Alternative: Use local HuggingFace Embeddings (No rate limits)
We will use `langchain-community` with `sentence-transformers` to generate embeddings locally on your Colab instance. Let's install the package first.

In [35]:
!pip install sentence-transformers

In [36]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Using a popular and highly efficient local embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_2657/384384360.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**2. Store embeddings in a vector database  **

In [37]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

**3. Implement similarity search mechanism **

In [38]:
query="What all are the ML and DL concepts are explained in the document?"

In [39]:
results=vector_store.similarity_search_with_score(query=query)

for i, (doc,score) in enumerate(results):
  print(f"RESULT{i}")
  print(f"Score:{score}")
  print(f"Document:{doc.page_content[:500]}")

RESULT0
Score:1.1163947582244873
Document:4 Discussing all of them is beyond the scope of the book, and we refer you to the scikit-learn documentation
for more details.
view of how each algorithm builds a model. We will examine the strengths and weak‐
nesses of each algorithm, and what kind of data they can best be applied to. We will
also explain the meaning of the most important parameters and options. 4 Many algo‐
rithms have a classification and a regression variant, and we will describe both.
It is not necessary to read through the
RESULT1
Score:1.117124080657959
Document:• Chapter 8 offers a high-level overview, and includes references to more advanced
topics.
While Chapters 2 and 3 provide the actual algorithms, understanding all of these
algorithms might not be necessary for a beginner. If you need to build a machine
learning system ASAP, we suggest starting with Chapter 1 and the opening sections of
Chapter 2, which introduce all the core concepts. You can then skip to “Summar

### **Part 3 — Deliverables and Explanation**

Below is the documentation detailing our approach, the embedding model, the vector database, and the results from our sample query.

---

#### **1. Description of the Embedding Model Used**
* **Model**: `all-MiniLM-L6-v2` via HuggingFace's `sentence-transformers` library.
* **Overview**: This is a highly efficient, compact sentence-transformer model that maps sentences and paragraphs to a **384-dimensional dense vector space**.
* **Why it was chosen**:
  * **Performance & Speed**: It is optimized for speed and is light enough to run locally inside Google Colab CPU runtime without needing external API dependencies or incurring rate limits (such as the `429 Resource Exhausted` error from the Gemini API).
  * **Accuracy**: Despite its small size, it performs exceptionally well on semantic search tasks, grouping concepts with highly related meanings close together in the vector space.

---

#### **2. Explanation of the Vector Storage Approach**
* **Technology**: **FAISS (Facebook AI Similarity Search)** via `langchain_community.vectorstores`.
* **Overview**: FAISS is a library developed by Meta for efficient similarity search and clustering of dense vectors. It is highly optimized for fast nearest-neighbor lookups in high-dimensional spaces.
* **Implementation Details**:
  1. **Chunking**: The text extracted from the 392-page PDF document was divided into smaller, contextual segments using `RecursiveCharacterTextSplitter` with a `chunk_size` of 1000 characters and a `chunk_overlap` of 150 characters. This overlap ensures context is preserved across split boundaries.
  2. **Index Construction**: FAISS processes the generated 384-dimensional embeddings and indexes them using L2 (Euclidean) distance to calculate semantic similarity.
  3. **Retrieval**: When a query is made, FAISS converts the query into the same 384-dimensional embedding space and uses vector math to find the closest document chunks, returning them alongside an L2 distance score (lower scores indicate higher similarity).

#### **3. Example of Retrieved Chunks for a Sample Query**

* **Query**: *"What all are the ML and DL concepts are explained in the document?"*

##### **Top Retrieved Chunks and Relevance:**

1. **Result 0 (Score: 1.116)**
   * **Snippet**: `"Discussing all of them is beyond the scope of the book, and we refer you to the scikit-learn documentation for more details. view of how each algorithm builds a model. We will examine the strengths and weaknesses..."`
   * **Relevance**: Identifies that the source document focuses heavily on standard Machine Learning algorithms (supervised classification and regression) built around the `scikit-learn` ecosystem.

2. **Result 1 (Score: 1.117)**
   * **Snippet**: `"If you need to build a machine learning system ASAP, we suggest starting with Chapter 1 and the opening sections of Chapter 2, which introduce all the core concepts. You can then skip to 'Summary and Outlook'..."`
   * **Relevance**: Directly points the user to the chapters containing the core Machine Learning concepts and supervised learning frameworks covered in the book.

3. **Result 2 (Score: 1.187)**
   * **Snippet**: `"section devoted to it for details. Then you can use the techniques in Chapter 5 to evaluate and tune your model..."`
   * **Relevance**: Highlights crucial ML workflow concepts like model evaluation, validation, and hyperparameter tuning.

4. **Result 3 (Score: 1.205)**
   * **Snippet**: `"multiclass classification, 63 | ordinary least squares, 47 | ridge regression, 49 | linear support vector machines (SVMs), 56 | loss functions, 56 | machine learning algorithm chains and pipelines, 305-321..."`
   * **Relevance**: Extracts actual concrete ML algorithms and keywords covered in the book index, showing topics like multiclass classification, linear regression, SVMs, pipelines, and loss functions.

**Part 3 — Retrieval Pipeline**

**1. Accept user query **

In [40]:
retriever=vector_store.as_retriever(
    search_type="similarity", #performing similarity search b/w vector embeddings and query embeddings
    search_kwargs={"k":1} #generating top 1 response
)

In [48]:
query1="What all are the learning concepts are explained in the document?"

In [49]:
results=vector_store.similarity_search_with_score(query=query1)

for i, (doc,score) in enumerate(results):
  print(f"RESULT{i}")
  print(f"Score:{score}")
  print(f"Document:{doc.page_content[:500]}")

RESULT0
Score:1.0794456005096436
Document:4 Discussing all of them is beyond the scope of the book, and we refer you to the scikit-learn documentation
for more details.
view of how each algorithm builds a model. We will examine the strengths and weak‐
nesses of each algorithm, and what kind of data they can best be applied to. We will
also explain the meaning of the most important parameters and options. 4 Many algo‐
rithms have a classification and a regression variant, and we will describe both.
It is not necessary to read through the
RESULT1
Score:1.091812252998352
Document:• Chapter 8 offers a high-level overview, and includes references to more advanced
topics.
While Chapters 2 and 3 provide the actual algorithms, understanding all of these
algorithms might not be necessary for a beginner. If you need to build a machine
learning system ASAP, we suggest starting with Chapter 1 and the opening sections of
Chapter 2, which introduce all the core concepts. You can then skip to “Summar

In [50]:
retriever=vector_store.as_retriever(
    search_type="similarity", #performing similarity search b/w vector embeddings and query embeddings
    search_kwargs={"k":1} #generating top 1 response
)

### **2. Retrieve Top-K Relevant Chunks & Experiment with different values of K**
We will define a helper function to query our vector store and retrieve the top `k` chunks, then test this with $k = 3$, $k = 5$, and $k = 7$.

In [51]:
def retrieve_chunks(query, k):
    """
    Retrieves the top-k relevant chunks from the FAISS vector store for a given query.
    """
    # Create a retriever dynamically with the specified k
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    # Retrieve relevant documents
    retrieved_docs = retriever.invoke(query)
    return retrieved_docs

# Define our test query
test_query = "What are the main machine learning models explained in this book?"

# Experiment with different values of k
for k_value in [3, 5, 7]:
    print("=" * 80)
    print(f"EXPERIMENTING WITH k = {k_value}")
    print("=" * 80)

    docs = retrieve_chunks(test_query, k=k_value)

    for idx, doc in enumerate(docs):
        page_num = doc.metadata.get('page', 'Unknown')
        print(f"\n--- Chunk {idx + 1} (Page {page_num}) ---")
        # Print first 250 characters of the chunk for comparison
        print(doc.page_content[:250].replace('\n', ' ') + "...")

EXPERIMENTING WITH k = 3

--- Chunk 1 (Page 9) ---
• Chapter 8 offers a high-level overview, and includes references to more advanced topics. While Chapters 2 and 3 provide the actual algorithms, understanding all of these algorithms might not be necessary for a beginner. If you need to build a machi...

--- Chunk 2 (Page 9) ---
how to use the large array of models already implemented in scikit-learn and other libraries. Why We Wrote This Book There are many books on machine learning and AI. However, all of them are meant for graduate students or PhD students in computer sci...

--- Chunk 3 (Page 43) ---
4 Discussing all of them is beyond the scope of the book, and we refer you to the scikit-learn documentation for more details. view of how each algorithm builds a model. We will examine the strengths and weak‐ nesses of each algorithm, and what kind ...
EXPERIMENTING WITH k = 5

--- Chunk 1 (Page 9) ---
• Chapter 8 offers a high-level overview, and includes references to more advanced 

**Part 4 — Answer Generation (RAG)**

In [52]:
from langchain_core.prompts import ChatPromptTemplate

In [53]:
rag_prompt=ChatPromptTemplate.from_template(
    """
    You are a Senior ML & Data Analyst and your job is to answer
    the user questions based on the retrieved context.

    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved Context:{context}
    Question:{question}

    Provide:
    1. Answer
    2. Supporting Evidence
    """
)

In [54]:

from langchain_core.output_parsers import StrOutputParser

In [55]:
rag_chain=rag_prompt|llm|StrOutputParser()

In [58]:
question="Briefly summarize the learning concepts in the document?"

retrieved_docs=retriever.invoke(question)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response=rag_chain.invoke({
    "context":context,
    "question":question,
    }
)

In [59]:
print(response)

**1. Answer**
The document highlights several key learning concepts in text processing, natural language processing (NLP), and machine learning, including:
* **Fundamental Algorithms:** Core concepts in information retrieval, NLP, and machine learning (referenced in Manning et al.'s *Introduction to Information Retrieval*).
* **Basic Text Processing:** Implementing simple feature extraction methods using `CountVectorizer` and `TfidfVectorizer`.
* **Advanced Text-Processing Libraries & Topic Modeling:** Utilizing specialized Python packages such as `spacy` (efficient text processing), `nltk` (established NLP tools), and `gensim` (focused on topic modeling).
* **Neural Network Developments:** Modern advancements in text processing, specifically the use of continuous vector representations (word vectors).

---

**2. Supporting Evidence**
* "...describes fundamental algorithms in information retrieval, NLP, and machine learning."
* "...the classes `CountVectorizer` and `TfidfVectorizer` on

### **Part 4 — Deliverables and Explanation (Answer Generation)**

Below is the documentation detailing our prompt design, sample queries, and the strict evidence of grounding implemented in our RAG system.

---

#### **1. Prompt Design Explanation**
We designed our RAG prompt using a structured template tailored for a role-specific assistant:
```python
rag_prompt = ChatPromptTemplate.from_template(
    """
    You are a Senior ML & Data Analyst and your job is to answer
    the user questions based on the retrieved context.

    If the answer is not available or cannot be found in the context then say - "I couldn't find this information"

    Retrieved Context:{context}
    Question:{question}

    Provide:
    1. Answer
    2. Supporting Evidence
    """
)
```
* **Persona Setting**: Setting the system role as a *Senior ML & Data Analyst* ensures that the output language is precise, educational, and professionally structured.
* **Strict Anti-Hallucination Constraints**: By specifying *"If the answer is not available or cannot be found in the context then say - 'I couldn't find this information'"*, we restrict the model from drawing on pre-trained external knowledge. It must remain strictly faithful to the parsed pages of `/content/intro-to-ml.pdf`.
* **Structured Output (1. Answer, 2. Supporting Evidence)**: This guarantees that every answer is parsed into a clean summary followed by verifiable reference text, allowing users to verify facts manually.

---

#### **2. Sample Queries and Generated Answers**

* **Query**: *"Briefly summarize the learning concepts in the document?"*
* **Generated Answer**:
  * The document covers fundamental algorithms in information retrieval, NLP, and machine learning.
  * It touches upon basic text-processing features via `CountVectorizer` and `TfidfVectorizer`.
  * It references advanced text-processing tools (such as `spacy`, `nltk`, and `gensim`).
  * It highlights recent neural network developments such as continuous vector representations (word vectors).

---

#### **3. Evidence of Grounding (Context-Based Answers)**
To prove that the generated response is entirely grounded in the retrieved textbook segments rather than the LLM's imagination, the model extracted the exact citations from the context:
* **Source Context Citation 1**: `"...describes fundamental algorithms in information retrieval, NLP, and machine learning."`
* **Source Context Citation 2**: `"...the classes CountVectorizer and TfidfVectorizer only implement relatively simple text-processing methods."`
* **Source Context Citation 3**: `"For more advanced text-processing methods, we recommend the Python packages spacy [...], nltk [...], and gensim..."`
* **Source Context Citation 4**: `"There have been several very exciting new developments in text processing in recent years [...] and relate to neural networks. The first is the use of continuous vector representations, also known as word vectors..."`

**Part 5 — End-to-End RAG Application **

In [60]:
question2="help me understand the neural network concept"

retrieved_docs=retriever.invoke(question2)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response=rag_chain.invoke({
    "context":context,
    "question":question2,
    }
)

In [61]:

print(response)

**1. Answer**

Neural networks, also known as deep learning, are a family of algorithms used in machine learning applications. Simple neural network methods include Multilayer Perceptrons (MLPs), also referred to as (vanilla) feed-forward neural networks, which are used for classification and regression tasks. Conceptually, MLPs can be understood as generalizations of linear models that reach a decision by performing multiple stages of processing. Additionally, deep learning algorithms are often carefully tailored to specific use cases.

**2. Supporting Evidence**

* *"A family of algorithms known as neural networks has recently seen a revival under the name “deep learning.”"*
* *"While deep learning shows great promise in many machine learning applications, deep learning algorithms are often tailored very carefully to a specific use case."*
* *"Here, we will only discuss some relatively simple methods, namely multilayer perceptrons for classification and regression... Multilayer perce

In [64]:
question3="help me understand the Bag-of-Words representations concept"

retrieved_docs=retriever.invoke(question3)

context="\n\n".join(
    [doc.page_content for doc in retrieved_docs])

response=rag_chain.invoke({
    "context":context,
    "question":question3,
    }
)

In [65]:

print(response)

### 1. Answer

The **Bag-of-Words** representation is a method for processing text where the structure of the text is discarded, and the text is represented simply by counting the occurrences of words (as if placing the words into a "bag"). 

Computing a Bag-of-Words representation for a collection (corpus) of documents involves three core steps:

1. **Tokenization**: Splitting each document into individual words (called tokens), such as by breaking text at whitespace and punctuation.
2. **Vocabulary Building**: Gathering a list of all unique words that appear across any of the documents and assigning a number to each (e.g., in alphabetical order).
3. **Encoding**: Counting how frequently each word from the built vocabulary appears within each specific document.

The final output of this process is a vector of word counts for each document.

---

### 2. Supporting Evidence

The provided context directly details this concept through the following text:

* *"Discarding the structure and 